In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from simulation.eg_parameters import Parameters
from simulation.eg_runner_usingMonitoredResource import RunnerMR
#from simulation.eg_model import Model
import os
from IPython.display import HTML
from itables import to_html_datatable


results = []
# Loop through doctor counts 3-6, running the simulation
for i in range(3, 7):
    param = Parameters(number_of_doctors=i)
    runner = RunnerMR(param=param)
    result = runner.run_reps()["run"]
    # Add scenario information to the results, then save to list
    result["number_of_doctors"] = i
    results.append(result)

# Combine results into a single dataframe
scenario_results = pd.concat(results, ignore_index=True)

# Show as interactive table
HTML(to_html_datatable(scenario_results))

In [4]:
import itertools


def run_scenarios(scenarios, param_factory=None):
    """
    Execute a set of scenarios and return the results from each run.

    Parameters
    ----------
    scenarios : dict
        Dictionary where key is name of parameter and value is a list with
        different values to run in scenarios.
    param_factory : callable or None, optional
        A callable that returns a new Parameters object for each scenario run.
        This can be a class (e.g., `Parameters`) or a factory function/lambda
        with preset arguments (e.g., `lambda: Parameters(number_of_runs=4)`).
        If not provided, defaults to using `Parameters()` with no arguments.

    Returns
    -------
    pandas.DataFrame
        DataFrame with results from each run of each scenario.

    Notes
    -----
    Function adapted from Rosser, Chalk and Heather 2025.
    """
    # If none provided, use Parameters
    if param_factory is None:
        param_factory = Parameters

    # Find every possible permutation of the scenarios
    all_scenarios_tuples = list(itertools.product(*scenarios.values()))

    # Convert back into dictionaries
    all_scenarios_dicts = [
        dict(zip(scenarios.keys(), p)) for p in all_scenarios_tuples
    ]

    # Preview some of the scenarios
    print(f"There are {len(all_scenarios_dicts)} scenarios. Running:")

    # Run the scenarios...
    results = []
    for index, scenario_to_run in enumerate(all_scenarios_dicts):
        print(scenario_to_run)

        # Create fresh instance of parameter class for each scenario
        param = param_factory()

        # Update parameter list with the scenario parameters
        param.scenario_name = index
        for key in scenario_to_run:
            setattr(param, key, scenario_to_run[key])

        # Perform replications
        scenario_exp = RunnerMR(param)
        scenario_res = scenario_exp.run_reps()["run"]

        # Add scenario number and values to the results dataframe
        scenario_res["scenario"] = index
        for key in scenario_to_run:
            scenario_res[key] = scenario_to_run[key]

        # Add results from scenario to list
        results.append(scenario_res)
    return pd.concat(results)

In [ ]:
# Run scenarios
scenario_results = run_scenarios(
    scenarios={"interarrival_time": [4, 5, 6, 7, 8],
               "number_of_doctors": [3, 4, 5]},
    param_factory=Parameters
)

In [ ]:
# Save to CSV

import os

os.makedirs("scenarios_resources", exist_ok=True)

scenario_results.to_csv(
    "scenarios_resources/python_scenario_results.csv", index=False
)

# View scenario results
HTML(to_html_datatable(scenario_results.head(200)))

In [ ]:
# Run sensitivity analysis
sensitivity_results = run_scenarios(
    scenarios={"interarrival_time": [4, 4.5, 5, 5.5, 6, 6.5, 7]},
    param_factory=Parameters
)

In [ ]:
# Save to CSV
sensitivity_results.to_csv(
    "scenarios_resources/python_sensitivity_results.csv", index=False
)

# View sensitivity results
HTML(to_html_datatable(sensitivity_results.head(200)))

In [ ]:
scenario = pd.read_csv("scenarios_resources/python_scenario_results.csv")
HTML(to_html_datatable(scenario.head(200)))

In [20]:
sensitivity = pd.read_csv("scenarios_resources/python_sensitivity_results.csv")
HTML(to_html_datatable(sensitivity.head(200)))

Loading ITables v2.6.2 from the internet... (need help?)


In [17]:
import scipy.stats as st
import numpy as np


def summary_stats(data):
    """
    Calculate mean, standard deviation and 95% confidence interval (CI).

    Parameters
    ----------
    data : pd.Series
        Data to use in calculation.

    Returns
    -------
    tuple
        (mean, standard deviation, CI lower, CI upper).
    """
    # Remove any NaN from the series
    data = data.dropna()

    # Find number of observations
    count = len(data)

    # If there are no observations, then set all to NaN
    if count == 0:
        mean, std_dev, ci_lower, ci_upper = np.nan, np.nan, np.nan, np.nan
    # If there is only one or two observations, can do mean but not others
    elif count < 3:
        mean = data.mean()
        std_dev, ci_lower, ci_upper = np.nan, np.nan, np.nan
    # With more than one observation, can calculate all...
    else:
        mean = data.mean()
        std_dev = data.std()
        # Special case for CI if variance is 0
        if np.var(data) == 0:
            ci_lower, ci_upper = mean, mean
        else:
            # Calculation of CI uses t-distribution, which is suitable for
            # smaller sample sizes (n<30)
            ci_lower, ci_upper = st.t.interval(
                confidence=0.95,
                df=count-1,
                loc=mean,
                scale=st.sem(data))
    return mean, std_dev, ci_lower, ci_upper

def summarise_scenarios(results, groups, result_vars, path_prefix=None):
    """
    Find the average results from each scenario for multiple metrics.

    Parameters
    ----------
    results : pd.DataFrame
        Run-level results from every scenario.
    groups : list
        List of columns to group by (provided as strings).
    result_vars : list
        List of performance measures to get results on (provided as strings).
    path_prefix : str, optional
        Path prefix to save tables to. Each metric will be saved as
        {path_prefix}_{metric}.csv

    Returns
    -------
    summary_tables : dict
        Dictionary with metric names as keys and summary DataFrames as values.
    """
    summary_tables = {}

    for result_var in result_vars:
        summary_table = (
            results.groupby(groups)[result_var]
            .apply(summary_stats)
            .apply(pd.Series)
            .reset_index()
        )
        summary_table.columns = (
            list(summary_table.columns[:-4]) +
            ["mean", "std_dev", "ci_lower", "ci_upper"]
        )

        # Add column to identify which metric this is
        summary_table["metric"] = result_var

        summary_tables[result_var] = summary_table

        # Save if path provided
        if path_prefix:
            output_path = f"{path_prefix}_{result_var}.csv"
            summary_table.to_csv(output_path, index=False)

    return summary_tables

In [11]:
import os
os.makedirs("tables_figures_resources", exist_ok=True)

result_variables = [
    "mean_wait_time",
    "mean_utilisation_tw",
    "mean_queue_length",
    "mean_time_in_system",
    "mean_patients_in_system"
]

scenario_tables = summarise_scenarios(
    results=scenario,
    groups=["scenario", "interarrival_time", "number_of_doctors"],
    result_vars=result_variables,
    path_prefix=os.path.join("tables_figures_resources", "python_scenario")
)

In [ ]:
HTML(to_html_datatable(scenario_tables["mean_wait_time"]))

In [ ]:
HTML(to_html_datatable(scenario_tables["mean_utilisation_tw"]))

In [80]:
HTML(to_html_datatable(scenario_tables["mean_queue_length"]))

Loading ITables v2.6.2 from the internet... (need help?)


In [ ]:
HTML(to_html_datatable(scenario_tables["mean_time_in_system"]))

In [ ]:
HTML(to_html_datatable(scenario_tables["mean_patients_in_system"]))

In [21]:
sensitivity_tables = summarise_scenarios(
    results=sensitivity,
    groups=["scenario", "interarrival_time"],
    result_vars=result_variables,
    path_prefix=os.path.join("tables_figures_resources", "python_sensitivity")
)

In [22]:
HTML(to_html_datatable(sensitivity_tables["mean_wait_time"]))

Loading ITables v2.6.2 from the internet... (need help?)


In [86]:
HTML(to_html_datatable(sensitivity_tables["mean_utilisation_tw"]))

Loading ITables v2.6.2 from the internet... (need help?)


In [87]:
HTML(to_html_datatable(sensitivity_tables["mean_queue_length"]))

Loading ITables v2.6.2 from the internet... (need help?)


In [88]:
HTML(to_html_datatable(sensitivity_tables["mean_time_in_system"]))

Loading ITables v2.6.2 from the internet... (need help?)


In [89]:
HTML(to_html_datatable(sensitivity_tables["mean_patients_in_system"]))

Loading ITables v2.6.2 from the internet... (need help?)


In [23]:
import matplotlib.colors as mcolors

def pale_colour(colour, alpha=0.2):
    """
    Make pale version of a colour.

    Parameters
    ----------
    colour : str or tuple
        The colour to pale, as a Plotly-compatible name ('blue', 'red'), hex
        string (e.g., "#1f77b4"), or RGB tuple (e.g., (0.1, 0.3, 0.8)).
        Accepts any format readable by matplotlib.colors.to_rgb.
    alpha : float, optional
        Opacity for the pale colour (default = 0.2). Should be between 0 and 1.

    Returns
    -------
    str
        Colour in 'rgba(r, g, b, a)' string format
        (e.g., "rgba(31,119,180,0.2)").

    Acknowledgements
    ----------------
    This function was generated by Perplexity.
    """
    # Convert any accepted colour format to an RGB tuple
    rgb = mcolors.to_rgb(colour)
    # Scale values to 0-255 for plotly compatability
    r, g, b = [int(x * 255) for x in rgb]
    # Return as RGB tuple with added alpha parameter (which makes it paler)
    return f"rgba({r},{g},{b},{alpha})"

In [24]:
import plotly.express as px
import plotly.graph_objects as go

def plot_metrics(
    summary_tables, colour_var=None, name_mappings=None, path_prefix=None
):
    """
    Plot results from different model scenarios.

    Parameters
    ----------
    summary_tables : dict
        Dictionary with metric names as keys and summary DataFrames as values.
    colour_var : str, optional
        Name of variable to colour lines with.
    name_mappings : dict, optional
        Dictionary mapping column names to labels. If not provided,
        function will default to variable names.
    path : str
        Path to save figure to.

    Returns
    -------
    figs : dict
        Dictionary with metric names as keys and plotly figures as values.
    """
    figs = {}

    for metric_name, summary_table in summary_tables.items():

        # Initialise figure
        fig = go.Figure()

        # Handling color mappings (optional)
        if colour_var:
            unique_groups = summary_table[colour_var].unique()
            colours = px.colors.qualitative.Plotly
            colour_map = {
                group: colours[i % len(colours)]
                for i, group in enumerate(unique_groups)
            }
        else:
            colour_map = {None: "blue"}

        # Plot each scenario line and its shaded confidence interval
        if colour_var:
            groups = summary_table.groupby(colour_var)
        else:
            groups = [(None, summary_table)]
        for group_name, group_data in groups:
            x = group_data["interarrival_time"].values
            mean = group_data["mean"].values
            ci_upper = group_data["ci_upper"].values
            ci_lower = group_data["ci_lower"].values
            main_colour = colour_map[group_name]
            shade_colour = pale_colour(main_colour, alpha=0.2)

            # Filled confidence interval region
            fig.add_trace(go.Scatter(
                x=np.concatenate([x, x[::-1]]),
                y=np.concatenate([ci_upper, ci_lower[::-1]]),
                fill="toself",
                fillcolor=shade_colour,
                line={"color": "rgba(255,255,255,0)"},
                showlegend=False,
                name="95% CI",
                hoverinfo="name+y"
            ))

            # Mean line
            fig.add_trace(go.Scatter(
                x=x,
                y=mean,
                mode="lines",
                line={"color": main_colour, "width": 2},
                name=group_name,
                hoverinfo="y"
            ))

        # Set labels and layout
        fig.update_layout(
            xaxis_title=(
                name_mappings.get("interarrival_time", "interarrival_time")
                if name_mappings else "interarrival_time"
            ),
            yaxis_title=(
                name_mappings.get(metric_name, metric_name)
                if name_mappings else metric_name
            ),
            legend_title_text=(
                name_mappings.get(colour_var, colour_var)
                if colour_var and name_mappings
                else (colour_var if colour_var else None)
            ),
            template="plotly_white"
        )

        if not colour_var:
            fig.update_layout(showlegend=False)

        figs[metric_name] = fig

        # Save if path provided
        if path_prefix:
            output_path = f"{path_prefix}_{metric_name}.png"
            fig.write_image(output_path)

    return figs

In [26]:
name_mappings = {
    "interarrival_time": "Patient inter-arrival time",
    "number_of_doctors": "Number of doctors",
    "mean_wait_time": "Mean wait time for the doctor",
    "mean_utilisation_tw": "Mean utilisation",
    "mean_queue_length": "Mean queue length",
    "mean_time_in_system": "Mean patient time in the system",
    "mean_patients_in_system": "Mean number of patients in the system"
}



In [1]:
%pip install -U "plotly[kaleido]"

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------  9.7/9.9 MB 55.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 50.0 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 6.5.0
    Uninstalling plotly-6.5.0:
      Successfully uninstalled plotly-6.5.0
  Attempting uninstall: kaleido
    Found existing installation: kaleido 0.2.1
    Uninstalling kaleido-0.2.1:
      Successfully uninstalled kaleido-0.2.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip show kaleido

Name: kaleido
Version: 1.2.0
Summary: Plotly graph export library
Home-page: https://github.com/plotly/kaleido
Author: 
Author-email: Andrew Pikul <ajpikul@gmail.com>, Neyberson Atencio <neyberatencio@gmail.com>
License: The MIT License (MIT)

Copyright (c) Plotly, Inc

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTI

In [27]:
scenario_plots = plot_metrics(
    summary_tables=scenario_tables,
    colour_var="number_of_doctors",
    name_mappings=name_mappings,
    path_prefix=os.path.join("tables_figures_resources", "python_scenario")
)

In [28]:
scenario_plots["mean_wait_time"]

In [29]:
scenario_plots["mean_utilisation_tw"]

In [30]:
scenario_plots["mean_queue_length"]

In [31]:
scenario_plots["mean_patients_in_system"]

In [32]:
sensitivity_plots = plot_metrics(
    summary_tables=sensitivity_tables,
    name_mappings=name_mappings,
    path_prefix=os.path.join("tables_figures_resources", "python_sensitivity")
)

In [33]:
sensitivity_plots["mean_wait_time"]

In [34]:
sensitivity_plots["mean_utilisation_tw"]

In [35]:
sensitivity_plots["mean_queue_length"]

In [36]:
sensitivity_plots["mean_time_in_system"]

In [37]:
sensitivity_plots["mean_patients_in_system"]